#### Load Environment

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating new project at `~/Documents/GitHub/BEE_4750/bee_4750_final_project`
     Project No packages added to or removed from `~/Documents/GitHub/BEE_4750/bee_4750_final_project/Project.toml`
    Manifest No packages added to or removed from `~/Documents/GitHub/BEE_4750/bee_4750_final_project/Manifest.toml`


In [7]:
Pkg.add("JuMP")
Pkg.add("HiGHS")
Pkg.add("CSV")
Pkg.add("DataFrames")
Pkg.add("Statistics")
Pkg.add("Plots")
Pkg.add("Measures")

     Project No packages added to or removed from `~/Documents/GitHub/BEE_4750/bee_4750_final_project/Project.toml`
    Manifest No packages added to or removed from `~/Documents/GitHub/BEE_4750/bee_4750_final_project/Manifest.toml`
     Project No packages added to or removed from `~/Documents/GitHub/BEE_4750/bee_4750_final_project/Project.toml`
    Manifest No packages added to or removed from `~/Documents/GitHub/BEE_4750/bee_4750_final_project/Manifest.toml`
   Resolving package versions...
   Installed Durations ─ v1.4.1
   Installed CSV ─────── v1.3.0
    Updating `~/Documents/GitHub/BEE_4750/bee_4750_final_project/Project.toml`
  [336ed68f] + CSV v1.3.0
    Updating `~/Documents/GitHub/BEE_4750/bee_4750_final_project/Manifest.toml`
  [336ed68f] + CSV v1.3.0
  [9a962f9c] + DataAPI v1.16.0
  [48204bd6] + DataStrings v1.1.0
  [e2d170a0] + DataValueInterfaces v1.0.0
  [41ac09f5] + Durations v1.4.1
  [82899510] + IteratorInterfaceExtensions v1.0.0
  [2dfb63ee] + PooledArrays v1.4.3
  

In [8]:
using JuMP
using HiGHS
using CSV
using DataFrames
using Statistics
using Plots
using Measures

#### Load data

In [9]:
using CSV, DataFrames, Statistics

nonpoint = DataFrame(CSV.File(expanduser("fertManure.csv")))
point = DataFrame(CSV.File(expanduser("ptSource_WWTP_CSO.csv")))

# RIM stations
stations = ["01491000", "01578310", "01594440", 
            "01646580", "01668000", "01673000", 
            "01674500", "02035000", "02041650"]

# format station IDs
nonpoint.staid = lpad.(string.(nonpoint.staid), 8, '0')
point.staid = lpad.(string.(point.staid), 8, '0')

# filter for nine stations
nonpoint_df = filter(row -> row.staid in stations, nonpoint)
point_df = filter(row -> row.staid in stations, point)

# select columns
nonpoint_df = select(nonpoint_df, :staid, :year, :fertTN, :manureTN) # lbs N
point_df = select(point_df, :staid, :year, :ptsourcetn, :csotn) # lbs N

# calculate total nonpoint nitrogen
nonpoint_df.nonpoint_TN = nonpoint_df.fertTN .+ nonpoint_df.manureTN

# calculate total point nitrogen ASSUMING cso included
point_df.point_TN = point_df.ptsourcetn .+ point_df.csotn

# sort by station and year
sort!(nonpoint_df, [:staid, :year])
sort!(point_df, [:staid, :year])

# aggregate nonpoint TN by station (avg across years)
nonpoint_agg = combine(groupby(nonpoint_df, :staid),:nonpoint_TN => mean => :nonpoint_TN)

# aggregate point TN by station (avg across years)
point_agg = combine(groupby(point_df, :staid),:point_TN => mean => :point_TN)

# sort by station ID
sort!(nonpoint_agg, :staid)
sort!(point_agg, :staid)

#println(nonpoint_agg)
#println(point_agg)

# aggregate flow
flow = DataFrame(CSV.File("RIM_2023_AnnualLoadTable.csv"))
flow.STAID = lpad.(string.(flow.STAID), 8, '0')
flow_df = filter(row -> row.STAID in stations, flow)

# calculate average Q for each station
flow_agg = combine(groupby(flow_df, :STAID),:Q => mean => :Q) # lb/ft3

sort!(flow_agg, :STAID)
#println(flow_agg)

# available area for buffer for each staid
buffer = DataFrame(CSV.File("Buffer.csv"))
buffer.staid = lpad.(string.(buffer.staid), 8, '0')
get_A(station) = filter(row -> row.staid == station, buffer).available_area[1] # acres

# example use in optimization to retrieve available area for buffer for staid
# A_01491000 = get_A("01491000")

get_A (generic function with 1 method)

#### Show data

In [12]:
display(nonpoint_agg)

Row,staid,nonpoint_TN
,String,Float64
1,01491000,1.64986e6
2,01578310,1.36387e8
3,01594440,1.93245e6
4,01646580,8.6867e7
5,01668000,1.00097e7
6,01673000,4.20486e6
7,01674500,1.66073e6
8,02035000,1.60106e7
9,02041650,6.16584e6


In [13]:
display(point_agg)

Row,staid,point_TN
,String,Float64
1,01491000,385.343
2,01578310,2.31617e7
3,01594440,3.33001e5
4,01646580,3.5042e6
5,01668000,1.31003e5
6,01673000,85202.8
7,01674500,17640.6
8,02035000,1.72885e6
9,02041650,31400.5


In [14]:
display(flow_agg)

Row,STAID,Q
,String,Float64
1,01491000,152.759
2,01578310,40425.8
3,01594440,389.938
4,01646580,11916.7
5,01668000,1742.19
6,01673000,940.567
7,01674500,524.282
8,02035000,7077.96
9,02041650,1125.93


In [15]:
display(buffer)

Row,staid,drainage_area,available_area
,String,DataStri…,Int64
1,01491000,"72,320",3616
2,01578310,"17,344,000",867200
3,01594440,"222,720",11136
4,01646580,"7,404,800",370240
5,01668000,"1,020,800",51040
6,01673000,"689,920",34496
7,01674500,"385,920",19296
8,02035000,"4,001,280",200064
9,02041650,"858,880",42944


#### Constants

In [16]:
n_watersheds = 9                # number of watersheds
n_tiers = 4                     # number of WWTP tiers

# forest buffer constants
e = 0.4                         # nitrogen removal efficiency

0.4

## TN Optimization

In [ ]:
#### TO DO : CHECK FOR UNITS CONSISTENCY!!

# set up model
model = Model(HiGHS.Optimizer)
@variable(model, y[w in 1:n_watersheds] >= 0)                           # acres of forest buffers to plant
@variable(model, f[w in 1:n_watersheds, i in 1:n_tiers] >= 0)           # fraction of WWTP flow to treat at each tier
@expression(model, nonpoint, sum(nonpoint_agg[w,:nonpoint_TN] * (1 - (y[w] * e)/buffer[w, :available_area]) for w in 1:n_watersheds))
@expression(model, point, sum(missing for w in 1:n_watersheds))                     # need to think about this problem formulation
@expression(model, cost, sum(missing for w in 1:n_watershed))                       # to write
@objective(model, Min, nonpoint + point)
@constraint(model, buffer_availability[w in 1:n_watersheds], y[w] <= buffer[w, :available_area])
@constraint(model, treatment_capacity, missing)                                     # need to think about this problem formulation
@constraint(model, budget, missing)                                                 # need a budget DataFrame

# optimize the model
# set_silent(model)
# optimize!(model)

MethodError: MethodError: no method matching +(::AffExpr, ::Missing)
The function `+` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  +(::Any, ::Any, !Matched::Any, !Matched::Any...)
   @ Base operators.jl:642
  +(!Matched::Missing, ::Missing)
   @ Base missing.jl:122
  +(!Matched::Dates.AbstractTime, ::Missing)
   @ Dates ~/.julia/juliaup/julia-1.12.7+0.aarch64.apple.darwin14/Julia-1.12.app/Contents/Resources/julia/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:90
  ...


In [ ]:
# show results
total_tn = objective_value(model)
total_cost = value(cost)

display(DataFrame(Watershed_ID = stations, Buffer_Acres = value.(model[:y])))   # how to display treatment fraction??